In [1]:
mini_json_path = "/workspace/data/annotation/label/mini.json"

In [2]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. 环境适应性设置
plt.switch_backend('Agg')
sns.set_context("paper", font_scale=1.5)  # 设置为论文（paper）上下文模式
sns.set_style("ticks")

ann_map = {
    3: 1,
    4: 2,
    5: 3,
}
VALID_DIRECTIONS = {"up", "down", "left", "right"}
OUTPUT_DIR = "/workspace/data/annotation_analysis"


def optimize_visual_report(json_file, output_dir=OUTPUT_DIR):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 2. 数据解析
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    all_data = []
    for entry in data:
        ann_id = f"Person_{ann_map.get(entry.get('annotator'), entry.get('annotator'))}"
        video_name = os.path.basename(entry.get('video')).split('=')[-1]

        for label_item in entry.get("videoLabels", []):
            direction = str(label_item.get("timelinelabels", ["unknown"])[0]).strip().lower()
            if direction not in VALID_DIRECTIONS:
                continue

            for r in label_item.get("ranges", []):
                duration = r['end'] - r['start']
                all_data.append(
                    {
                        "Annotator": ann_id,
                        "Direction": direction,
                        "Duration": duration,
                        "Video": video_name,
                    }
                )

    df = pd.DataFrame(all_data)
    if df.empty:
        print("没有可用标签（仅保留 up/down/left/right 后为空）。")
        return

    # --- 绘图 A: 标注数量对比 (学术配色) ---
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x='Direction', hue='Annotator', palette='Set2')
    plt.title('Annotator Label Distribution (Count)', pad=20)
    plt.xlabel('Action Category')
    plt.ylabel('Number of Instances')
    sns.despine()  # 去掉上方和右方的边框，更美观
    plt.savefig(f'{output_dir}/count_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()

    # --- 维度 A: 标注频率差异 (Stacked Bar Chart) ---
    plt.figure(figsize=(10, 6))
    pivot_df = df.groupby(['Annotator', 'Direction']).size().unstack(fill_value=0)
    pivot_df = pivot_df.reindex(columns=['up', 'down', 'left', 'right'], fill_value=0)
    pivot_df.plot(kind='bar', stacked=True, ax=plt.gca(), colormap='tab10')
    plt.title('Annotator Labeling Preference (Count)', fontsize=14)
    plt.ylabel('Total Number of Labels')
    plt.legend(title='Directions', bbox_to_anchor=(1.05, 1))
    plt.xticks(rotation=0)
    sns.despine()
    plt.tight_layout()
    plt.savefig(f"{output_dir}/annotator_preference.png", dpi=300)
    plt.close()

    # --- 生成统计汇总表 (Pivot Table) ---
    summary = df.groupby(['Annotator', 'Direction']).agg({
        'Duration': ['count', 'mean', 'std']
    }).reset_index()
    # 扁平化表头
    summary.columns = ['Annotator', 'Direction', 'Count', 'Avg_Duration', 'Std_Dev']
    summary.to_csv(f'{output_dir}/statistical_summary.csv', index=False)

    print(f"仅统计方向 {sorted(VALID_DIRECTIONS)}")
    print(f"优化完成，图片和表格已保存在: {output_dir}/")


# 执行
optimize_visual_report(mini_json_path, OUTPUT_DIR)

仅统计方向 ['down', 'left', 'right', 'up']
优化完成，图片和表格已保存在: /workspace/data/annotation_analysis/


In [3]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Linux服务器环境支持
plt.switch_backend('Agg')

ann_map = {
    3: 1,
    4: 2,
    5: 3,
}
VALID_DIRECTIONS = {"up", "down", "left", "right"}
OUTPUT_DIR = "/workspace/data/annotation_analysis"


def visualize_with_front_fill(json_file, output_dir=OUTPUT_DIR + "/timeline_visuals"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 只保留上下左右 4 个方向
    color_map = {
        'up': '#E74C3C',
        'down': '#3498DB',
        'left': '#2ECC71',
        'right': '#F1C40F',
    }

    # 数据分组
    video_groups = {}
    for entry in data:
        raw_path = entry.get('video', 'unknown')
        video_name = raw_path.split('/')[-1] if '/' in raw_path else os.path.basename(raw_path)
        video_name = video_name.replace('.mp4', '')
        if video_name not in video_groups:
            video_groups[video_name] = []
        video_groups[video_name].append(entry)

    for video_name, entries in video_groups.items():
        plt.figure(figsize=(16, 8))

        annotators = sorted(list(set([str(ann_map[e.get('annotator')]) for e in entries])))
        ann_to_y = {ann_id: i for i, ann_id in enumerate(annotators)}

        for entry in entries:
            ann_id = str(ann_map[entry.get('annotator')])
            y_idx = ann_to_y[ann_id]

            # 提取该标注者所有的原始片段并排序
            all_segments = []
            for label_item in entry.get("videoLabels", []):
                label = str(label_item.get("timelinelabels", ["unknown"])[0]).strip().lower()
                if label not in VALID_DIRECTIONS:
                    continue
                for r in label_item.get("ranges", []):
                    all_segments.append({'start': r['start'], 'end': r['end'], 'label': label})

            # 按时间排序
            all_segments.sort(key=lambda x: x['start'])

            # 绘制所有片段（不再填充 front）
            for seg in all_segments:
                color = color_map.get(seg['label'], '#95A5A6')
                plt.hlines(y=y_idx, xmin=seg['start'], xmax=seg['end'], color=color, linewidth=30, alpha=1.0)

        # 4. 图表修饰
        plt.yticks(range(len(annotators)), [f"Annotator {a}" for a in annotators], fontsize=12)
        plt.xlabel("Timeline (Frames/ms)", fontsize=12)
        plt.title(f"Comparison with: {video_name}", fontsize=16, pad=20)
        plt.grid(axis='x', linestyle=':', alpha=0.5)
        plt.ylim(-0.5, len(annotators) - 0.5)

        # 凡例
        legend_patches = [mpatches.Patch(color=color, label=label) for label, color in color_map.items()]
        plt.legend(handles=legend_patches, title="Directions", bbox_to_anchor=(1.02, 1), loc='upper left')

        save_path = os.path.join(output_dir, f"{video_name}.png")
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close()
        print(f"Success: {video_name} visualization saved to {save_path}")


# 运行（请确保路径正确）
visualize_with_front_fill(mini_json_path, OUTPUT_DIR + "/timeline_visuals")

Success: person_01_day_high_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_01_day_high_h265.png
Success: person_01_day_low_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_01_day_low_h265.png
Success: person_01_night_high_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_01_night_high_h265.png
Success: person_01_night_low_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_01_night_low_h265.png
Success: person_02_day_high_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_02_day_high_h265.png
Success: person_02_day_low_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_02_day_low_h265.png
Success: person_02_night_high_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_02_night_high_h265.png


Success: person_02_night_low_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_02_night_low_h265.png
Success: person_03_day_high_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_03_day_high_h265.png
Success: person_03_day_low_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_03_day_low_h265.png
Success: person_03_night_high_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_03_night_high_h265.png
Success: person_03_night_low_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_03_night_low_h265.png
Success: person_04_day_high_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_04_day_high_h265.png
Success: person_04_day_low_h265 visualization saved to /workspace/data/annotation_analysis/timeline_visuals/person_04_day_low_h265.png
Success: person_04_night_high_h265 vi

In [4]:
from collections import Counter, defaultdict


def _extract_filtered_segments(entry, valid_directions):
    segments = []
    for label_item in entry.get("videoLabels", []):
        label = str(label_item.get("timelinelabels", ["unknown"])[0]).strip().lower()
        if label not in valid_directions:
            continue
        for r in label_item.get("ranges", []):
            start = float(r["start"])
            end = float(r["end"])
            if end <= start:
                continue
            segments.append((start, end, label))
    segments.sort(key=lambda x: x[0])
    return segments


def _label_at_time(segments, t):
    # segments 已按 start 排序
    for s, e, lb in segments:
        if s <= t < e:
            return lb
    return None


def _majority_label(votes):
    votes = [vote for vote in votes if vote is not None]
    if not votes:
        return None
    c = Counter(votes)
    m = max(c.values())
    cands = sorted([k for k, v in c.items() if v == m])
    return cands[0]


def evaluate_majority_vote_performance(json_file, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 按视频聚合
    by_video = defaultdict(list)
    for entry in data:
        video_name = os.path.basename(entry.get("video", "unknown")).split("=")[-1]
        by_video[video_name].append(entry)

    # 统计容器（按时长加权）
    ann_total_duration = defaultdict(float)
    ann_match_duration = defaultdict(float)
    label_duration = defaultdict(float)
    total_vote_duration = 0.0

    for video_name, entries in by_video.items():
        ann_entries = {}
        for e in entries:
            ann_raw = e.get("annotator")
            ann_id = f"Person_{ann_map.get(ann_raw, ann_raw)}"
            ann_entries[ann_id] = e

        if len(ann_entries) < 2:
            continue

        ann_segments = {}
        for ann_id, e in ann_entries.items():
            segs = _extract_filtered_segments(e, VALID_DIRECTIONS)
            if segs:
                ann_segments[ann_id] = segs

        if len(ann_segments) < 2:
            continue

        boundaries = set()
        for segs in ann_segments.values():
            for s, e, _ in segs:
                boundaries.add(float(s))
                boundaries.add(float(e))
        boundaries = sorted(boundaries)

        for i in range(len(boundaries) - 1):
            s = boundaries[i]
            e = boundaries[i + 1]
            if e <= s:
                continue
            mid = (s + e) / 2.0
            dur = e - s

            votes = []
            vote_by_ann = {}
            for ann_id, segs in ann_segments.items():
                lb = _label_at_time(segs, mid)
                if lb is not None:
                    votes.append(lb)
                    vote_by_ann[ann_id] = lb

            maj = _majority_label(votes)
            if maj is None:
                continue

            label_duration[maj] += dur
            total_vote_duration += dur

            for ann_id, lb in vote_by_ann.items():
                ann_total_duration[ann_id] += dur
                if lb == maj:
                    ann_match_duration[ann_id] += dur

    if not ann_total_duration:
        print("没有可计算多数投票表现的数据。")
        return

    # 汇总表
    rows = []
    for ann_id in sorted(ann_total_duration.keys()):
        total_d = ann_total_duration[ann_id]
        match_d = ann_match_duration[ann_id]
        acc = (match_d / total_d * 100.0) if total_d > 0 else 0.0
        rows.append(
            {
                "Annotator": ann_id,
                "Total_Duration": total_d,
                "Agree_Duration": match_d,
                "Agree_Rate(%)": acc,
            }
        )

    perf_df = pd.DataFrame(rows).sort_values("Annotator")
    perf_csv = os.path.join(output_dir, "majority_vote_performance.csv")
    perf_df.to_csv(perf_csv, index=False)

    overall_acc = (
        perf_df["Agree_Duration"].sum() / perf_df["Total_Duration"].sum() * 100.0
    )

    print("=== 多数投票后表现（按时长加权）===")
    display(perf_df)
    print(f"Overall agreement to majority: {overall_acc:.2f}%")
    print(f"CSV saved: {perf_csv}")

    # 图1：每位标注者与多数投票一致率
    plt.figure(figsize=(8, 5))
    plt.bar(perf_df["Annotator"], perf_df["Agree_Rate(%)"], color="#4c78a8")
    plt.ylim(0, 100)
    plt.ylabel("Agreement Rate (%)")
    plt.title("Agreement with Majority Vote by Annotator")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    fig1 = os.path.join(output_dir, "majority_agreement_by_annotator.png")
    plt.savefig(fig1, dpi=300)
    plt.close()

    # 图2：多数投票后标签时长分布
    label_order = ["up", "down", "left", "right"]
    vals = [label_duration.get(lb, 0.0) for lb in label_order]
    plt.figure(figsize=(8, 5))
    plt.bar(label_order, vals, color=["#E74C3C", "#3498DB", "#2ECC71", "#F1C40F"])
    plt.ylabel("Duration")
    plt.title("Majority-Vote Label Duration Distribution")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    fig2 = os.path.join(output_dir, "majority_label_duration_distribution.png")
    plt.savefig(fig2, dpi=300)
    plt.close()

    print(f"Figure saved: {fig1}")
    print(f"Figure saved: {fig2}")


# 执行多数投票表现评估
evaluate_majority_vote_performance(mini_json_path, OUTPUT_DIR)

=== 多数投票后表现（按时长加权）===


,Annotator,Total_Duration,Agree_Duration,Agree_Rate(%)
0,Person_1,160183.0,155049.0,96.794916
1,Person_2,140553.0,136867.0,97.377502
2,Person_3,193354.0,187462.0,96.952740


Overall agreement to majority: 97.02%
CSV saved: /workspace/data/annotation_analysis/majority_vote_performance.csv
Figure saved: /workspace/data/annotation_analysis/majority_agreement_by_annotator.png
Figure saved: /workspace/data/annotation_analysis/majority_label_duration_distribution.png


In [5]:
def summarize_majority_vote_label_distribution(json_file, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    by_video = defaultdict(list)
    for entry in data:
        video_name = os.path.basename(entry.get("video", "unknown")).split("=")[-1]
        by_video[video_name].append(entry)

    label_duration = defaultdict(float)
    label_segment_count = defaultdict(int)
    total_duration = 0.0
    total_segments = 0

    for video_name, entries in by_video.items():
        ann_entries = {}
        for e in entries:
            ann_raw = e.get("annotator")
            ann_id = f"Person_{ann_map.get(ann_raw, ann_raw)}"
            ann_entries[ann_id] = e

        if len(ann_entries) < 2:
            continue

        ann_segments = {}
        for ann_id, e in ann_entries.items():
            segs = _extract_filtered_segments(e, VALID_DIRECTIONS)
            if segs:
                ann_segments[ann_id] = segs

        if len(ann_segments) < 2:
            continue

        boundaries = set()
        for segs in ann_segments.values():
            for s, e, _ in segs:
                boundaries.add(float(s))
                boundaries.add(float(e))
        boundaries = sorted(boundaries)

        for i in range(len(boundaries) - 1):
            s = boundaries[i]
            e = boundaries[i + 1]
            if e <= s:
                continue

            mid = (s + e) / 2.0
            dur = e - s
            votes = []
            for segs in ann_segments.values():
                lb = _label_at_time(segs, mid)
                if lb is not None:
                    votes.append(lb)

            maj = _majority_label(votes)
            if maj is None:
                continue

            label_duration[maj] += dur
            label_segment_count[maj] += 1
            total_duration += dur
            total_segments += 1

    if total_segments == 0 or total_duration == 0:
        print("没有可计算的多数投票分布数据。")
        return

    label_order = ["up", "down", "left", "right"]
    rows = []
    for label in label_order:
        dur = label_duration.get(label, 0.0)
        seg_count = label_segment_count.get(label, 0)
        rows.append(
            {
                "Label": label,
                "Duration": dur,
                "Duration_Ratio(%)": dur / total_duration * 100.0,
                "Segment_Count": seg_count,
                "Segment_Ratio(%)": seg_count / total_segments * 100.0,
            }
        )

    dist_df = pd.DataFrame(rows)
    dist_csv = os.path.join(output_dir, "majority_vote_label_distribution.csv")
    dist_df.to_csv(dist_csv, index=False)

    print("=== 多数投票后的标签分布 ===")
    display(dist_df)
    print(f"Total duration: {total_duration:.1f}")
    print(f"Total segments: {total_segments}")
    print(f"CSV saved: {dist_csv}")

    fig, ax1 = plt.subplots(figsize=(9, 5))
    bars = ax1.bar(
        dist_df["Label"],
        dist_df["Duration_Ratio(%)"],
        color=["#E74C3C", "#3498DB", "#2ECC71", "#F1C40F"],
    )
    ax1.set_ylabel("Duration Ratio (%)")
    ax1.set_title("Majority-Vote Label Distribution")
    ax1.grid(axis="y", alpha=0.25)
    ax1.set_ylim(0, max(100.0, float(dist_df["Duration_Ratio(%)"].max()) * 1.15))

    for bar, pct in zip(bars, dist_df["Duration_Ratio(%)"]):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=10,
        )

    plt.tight_layout()
    fig_path = os.path.join(output_dir, "majority_vote_label_distribution.png")
    plt.savefig(fig_path, dpi=300)
    plt.close()

    print(f"Figure saved: {fig_path}")


# 执行多数投票后的标签分布统计
summarize_majority_vote_label_distribution(mini_json_path, OUTPUT_DIR)

=== 多数投票后的标签分布 ===


,Label,Duration,Duration_Ratio(%),Segment_Count,Segment_Ratio(%)
0,up,18194.0,7.750967,866,5.563765
1,down,77062.0,32.829780,6789,43.617090
2,left,71541.0,30.477736,4789,30.767748
3,right,67935.0,28.941516,3121,20.051397


Total duration: 234732.0
Total segments: 15565
CSV saved: /workspace/data/annotation_analysis/majority_vote_label_distribution.csv
Figure saved: /workspace/data/annotation_analysis/majority_vote_label_distribution.png
